# Per-donor PhenoCycler islet trajectories

Each point is a native pancreatic islet. Start with donor 115, change the controls below, and run all cells. The feature tree follows differences in measured islet profiles. Seurat scSLIDE independently learns a curve from the distributions of selected proteins in the cells belonging to each islet.

The shared cohort score places donors on the same observed inflammation scale. The trees, diffusion coordinates and principal-curve distances are fit separately for each donor. They describe a cross-sectional ordering.

[Seurat fits for all 25 donors](../releases/phenocycler_gate_calibration_20261116/islet_trajectories/scslide/manifest.csv) · [Donor 115 Seurat coordinates](../releases/phenocycler_gate_calibration_20261116/islet_trajectories/scslide/115/curve.parquet) · [Measured donor paths (PDF)](../releases/phenocycler_gate_calibration_20261116/islet_trajectories/selected_paths/measured/donor_paths.pdf) · [Excess-zero-checked donor paths (PDF)](../releases/phenocycler_gate_calibration_20261116/islet_trajectories/selected_paths/excess_zero/donor_paths.pdf) · [Count-model diagnostics](../releases/phenocycler_gate_calibration_20261116/islet_trajectories/selected_paths/excess_zero/zero_model.csv)

In [ ]:
from pathlib import Path
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

locations = [Path.cwd(), *Path.cwd().parents]
if os.environ.get("PHENOCYCLER_DYNAMIC_RELEASE"):
    locations.insert(0, Path(os.environ["PHENOCYCLER_DYNAMIC_RELEASE"]).expanduser())
helper = next((folder for loc in locations for folder in (loc, loc / "scanpy", loc / "releases" / "phenocycler_gate_calibration_20261116" / "scanpy")
               if (folder / "islet_paths.py").is_file() and (folder / "release_data.py").is_file()), None)
if helper is None:
    raise FileNotFoundError("Open the release notebook beside its Python helpers or set PHENOCYCLER_DYNAMIC_RELEASE.")
sys.path.insert(0, str(helper))
import release_data as rd
import islet_paths as paths

DONOR = "115"
FIT_FEATURES = ["core.immune", "peri.immune"]
ORIENT_BY = "combined.immune"
ROOT_END = "low"                 # Tree anchor can be low or high.
PLOT_FEATURE = "core.protein.INS"
ZERO_INFLATION = True            # Model integer features with their area or cell-count exposure.
RUN_SCSLIDE = True
SCSLIDE_ZERO_INFLATION = False   # Check excess zeros in the islet-by-landmark count matrix.
SCSLIDE_PROTEINS = ["INS", "GCG", "SST", "CD3e", "CD68", "CD20"]
SCSLIDE_REGION = "core"           # Use "peri" for the donor-specific exterior rim.

## Choose islet features

The same feature names work for fitting, choosing the reference islet, and plotting. Change each independently. Prefix a feature with `core.`, `peri.`, or `combined.`. The combined value pools the numerators and exposures from both regions. Examples include `core.class.Beta`, `peri.lineage.Imm-T-cell`, `core.label.Macrophage_Gal3+`, `combined.protein.CD68`, and `core.positive.INS`.

Cell-type densities count cells assigned the selected label per mm². A zero can reflect a non-assessable typing gate. Protein means use cells with finite aligned intensity. Fitted-positive percentages use only cells with a decidable marker gate. A region without the requested measurement contributes a missing value. Inspect `catalog` to list all choices. Signed phenotype names and protein-symbol aliases such as `core.protein.LGALS3` and `core.positive.GZMB` are accepted alongside their stable keys.

In [ ]:
islets = paths.load_profiles(rd.BASE)
catalog = paths.feature_catalog(islets)
print(f"{islets.donor.nunique()} pancreas donors · {len(islets):,} native islets · {islets.assessed.sum():,} eligible")
print(f"{len(catalog):,} selectable features. Count-aware modeling applies to assigned cell densities and fitted-positive fractions.")
display(catalog.query("region == 'core'").groupby("measurement").size().rename("available features").to_frame())
display(catalog[catalog.feature.isin(["core.class.Beta", "core.lineage.Imm-NK-cell", "core.protein.M2Gal3", "core.positive.Granzyme_B"])][["feature","display_name","symbol_key"]])

## Fit the selected donor

Assessment requires **more than 10 eligible core cells**, at least one eligible rim cell, and positive measured tissue areas in both regions. The rim is one donor-specific median cell diameter wide, clipped to pancreatic tissue and neighboring islet or endocrine-cluster boundaries.

The graph connects distinct islet profiles by a minimum spanning tree in log-transformed feature space. Mixed feature families and continuous protein channels use one pooled scale per feature. Tied profiles share their distance. The root is the lowest or highest measured orientation feature among eligible fitted islets, with native UUID breaking ties. Absolute density and the pooled percentile remain available in separate columns. Positive values are ranked across all assessed islets. Measured zeros keep zero.

In [ ]:
run = paths.run_paths(islets, FIT_FEATURES, root_feature=ORIENT_BY, root_end=ROOT_END,
                      display_feature=PLOT_FEATURE, correction=ZERO_INFLATION, donors=[DONOR])
display(run.summary)
print("Eligibility and fit coverage")
display(run.points.path_status.value_counts().rename("islets").to_frame())
fig = paths.plot_donor(run, DONOR)
plt.show()

Panel A shows the fitted tree, colored by the chosen display feature's cohort percentile. With three or more fit features, its two-dimensional view uses principal components computed within the donor. The tree itself uses all selected dimensions. Panels B and C show measured feature values. Lines connect medians across ordered bins. Model-adjusted quantities are available in `run.fitted`; original values are in `run.raw`.

## Check excess zeros against exposure

For cell densities, the model uses integer assigned-cell counts with area in mm² as exposure. It compares a gamma–Poisson count model with a mixture that adds a point mass at zero. For a positive-marker fraction, it compares beta–binomial models with and without the extra zero component, using assessable cells as trials. Both use a common fit across assessed islets in the current cohort, then return rates or fractions in the original units.

The model uses the mixture only when it converges and improves BIC. A supported estimate is the posterior probability of membership in the count component times that component's posterior mean rate or fraction. An observed zero can acquire a positive modeled estimate. Cell calls, measured densities, and prevalence continue to use the observed counts. The extra-zero probability describes a statistical mixture; it cannot identify technical dropout versus biological absence in a section. Continuous intensities keep their measured values.

The table reports observed and expected zero counts, exposure range, estimated extra-zero probability and improvement in BIC. Positive improvement favors the mixture. Change `CHECK_FEATURE` to inspect a different count or marker. [Zero-inflated count models](https://doi.org/10.1080/00401706.1992.10485228) · [Negative binomial](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.nbinom.html) · [Beta–binomial](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.betabinom.html)

In [ ]:
CHECK_FEATURE = "core.class.Beta"
diagnostics = []
for feature in dict.fromkeys([*FIT_FEATURES, "combined.immune", CHECK_FEATURE]):
    measured = paths.measure(islets, feature)
    estimated, report = paths.zero_inflation(measured)
    diagnostics.append(report)
display(pd.DataFrame(diagnostics).set_index("feature").T)
checked = paths.measure(islets, CHECK_FEATURE)
estimated, report = paths.zero_inflation(checked)
donor_ids = run.points.loc[run.points.assessed].index
values = pd.DataFrame({"count": checked.numerator, "exposure": checked.exposure, "measured": checked.value,
                       "model_estimate": estimated}).loc[donor_ids].dropna().sort_values("exposure")
print(f"{paths.title(CHECK_FEATURE)} · {checked.units}. Examples are spread across the donor's exposure range.")
display(values.iloc[np.unique(np.linspace(0, len(values) - 1, min(6, len(values))).astype(int))] if len(values) else values)

## Run Seurat scSLIDE on native cell distributions

scSLIDE takes the measured protein vector of each cell in each assessed islet. A diffusion map and principal curve order the islets. The selected orientation feature determines the sign of arc length. A low absolute Spearman correlation means that orientation gives little information about progression along this protein curve. Programmatic outputs include each islet's complete-cell count, diffusion coordinates, curve coordinates, status and directed pseudotime.

This cell uses the release's R adapter and Seurat installation. Changing the donor, protein panel or region recomputes the fit locally. To work only with saved paths and the Python feature model, set `RUN_SCSLIDE = False` above. Protein channels absent from a donor's panel are unavailable. The display feature can be independent of the proteins used by scSLIDE.

Set `SCSLIDE_ZERO_INFLATION = True` to check for excess zeros in the per-islet **landmark counts** before fitting diffusion coordinates. The sampled landmark cells provide the reference. Each islet's remaining complete query cells supply the exposure for the zero model. The feature-tree correction above uses tissue area for immune-cell densities or assessable cells for marker positivity. The saved six-protein atlas uses the default unadjusted landmark matrix.

Install Seurat 6 and princurve with `install.packages(c("Seurat", "princurve"))` and destiny with `BiocManager::install("destiny")` in R. Set `PHENOCYCLER_R_LIB` for a nonstandard package library. [Seurat scSLIDE vignette](https://satijalab.org/seurat/articles/scslide_vignette)

In [ ]:
if RUN_SCSLIDE:
    from islet_scslide import run_scslide
    orientation = paths.measure(islets, ORIENT_BY).value
    sl = run_scslide(rd.BASE, DONOR, SCSLIDE_PROTEINS, orientation=orientation, region=SCSLIDE_REGION,
                     zero_adjust_landmarks=SCSLIDE_ZERO_INFLATION)
    print(f"{sl.status.eq('fitted').sum():,}/{len(sl):,} assessed islets fitted · "
          f"{sl.complete_cells.sum():,} complete cell–islet observations · "
          f"{sl.counted_cells.sum():,} non-landmark query observations")
    print(f"Orientation Spearman ρ = {sl.attrs['orientation_spearman']:.3f} for {paths.title(ORIENT_BY)}")
    print("Landmarks adjusted for excess zeros:", sl.attrs["adjusted_landmarks"])
    if SCSLIDE_ZERO_INFLATION:
        landmark_diagnostics = pd.DataFrame(sl.attrs["landmark_diagnostics"])
        display(landmark_diagnostics.reindex(columns=["feature", "mode", "raw_zeros", "expected_zeros_null",
                                                      "excess_zero_probability", "bic_improvement", "reason"]))
    print("Curve convergence:", sl.curve_converged.dropna().unique().tolist())
    fig = paths.plot_scslide(islets, sl, ORIENT_BY, PLOT_FEATURE, DONOR)
    plt.show()
    display(sl[["status", "complete_cells", "counted_cells", "dc1", "dc2", "pseudotime"]].head())
else:
    print("scSLIDE disabled in the controls above.")

## Export a selection

Set a destination to save your current donor and parameters. The default leaves the source data intact. The release's `islet_trajectories/selected_paths/` directory contains the measured and excess-zero-checked immune-feature trees for all donors, with graph edges, model diagnostics and a donor-page PDF for each mode.

In [ ]:
EXPORT_TO = None                 # Example: Path.home() / "islet-results" / DONOR
if EXPORT_TO is not None:
    destination = Path(EXPORT_TO).expanduser()
    destination.mkdir(parents=True, exist_ok=True)
    points = run.points.copy()
    for feature in dict.fromkeys([*FIT_FEATURES, ORIENT_BY, PLOT_FEATURE]):
        points[f"{feature}.measured"] = run.raw[feature]
        points[f"{feature}.model_input"] = run.fitted[feature]
    points.to_csv(destination / "islets.csv", index=False)
    run.edges.to_csv(destination / "edges.csv", index=False)
    run.summary.to_csv(destination / "fit.csv", index=False)
    pd.DataFrame(diagnostics).to_csv(destination / "zero_models.csv", index=False)
    paths.plot_donor(run, DONOR).savefig(destination / "feature_path.pdf")
    if RUN_SCSLIDE:
        sl.to_csv(destination / "scslide.csv", index_label="islet_id")
        paths.plot_scslide(islets, sl, ORIENT_BY, PLOT_FEATURE, DONOR).savefig(destination / "scslide.pdf")
    print(destination)